In [21]:
import pandas as pd

In [22]:
df = pd.read_csv("data/dataset_report_anonimizzati_llama_3_1_8B_4bit.csv")
df

,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,report_anonimo
0,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,"One year into the conflict, the risk of Famine...",[SHOCKS AND DRIVERS]: \nThe underlying causes ...
1,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Food insecurity remains alarmingly high in Afg...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
2,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,"During the 2017 post-harvest season, 33% of th...",[SHOCKS AND DRIVERS]: \nThe underlying causes ...
3,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,"Based on the September IPC analysis, it is exp...",[SHOCKS AND DRIVERS]: \nThe underlying causes ...
4,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,The results of this Acute Food Insecurity pilo...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
...,...,...,...,...,...,...,...
492,Haiti_Aug_2023_-_Jun_2024_KeyResults.txt,Haiti,Aug 2023 / Jun 2024,Aug 2023,Jun 2024,About 4.35 million people are experiencing hig...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
493,Djibouti_Mar_2022_-_Dec_2022_KeyResults.txt,Djibouti,Mar 2022 / Dec 2022,Mar 2022,Dec 2022,For the current analysis period of March throu...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
494,Djibouti_May_2013_-_May_2013_KeyResults.txt,Djibouti,May 2013 / May 2013,May 2013,May 2013,Food availability in the Republic of Djibouti ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
495,Madagascar_Sep_2024_-_Aug_2025_KeyResults.txt,Madagascar,Sep 2024 / Aug 2025,Sep 2024,Aug 2025,"Between September and December 2024, around 1....",[SHOCKS AND DRIVERS]: \nThe underlying causes ...


In [23]:
#df = pd.read_csv("df_duplicati.csv")
df = df.drop(columns="report_anonimo")
df

,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale
0,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,"One year into the conflict, the risk of Famine..."
1,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Food insecurity remains alarmingly high in Afg...
2,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,"During the 2017 post-harvest season, 33% of th..."
3,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,"Based on the September IPC analysis, it is exp..."
4,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,The results of this Acute Food Insecurity pilo...
...,...,...,...,...,...,...
492,Haiti_Aug_2023_-_Jun_2024_KeyResults.txt,Haiti,Aug 2023 / Jun 2024,Aug 2023,Jun 2024,About 4.35 million people are experiencing hig...
493,Djibouti_Mar_2022_-_Dec_2022_KeyResults.txt,Djibouti,Mar 2022 / Dec 2022,Mar 2022,Dec 2022,For the current analysis period of March throu...
494,Djibouti_May_2013_-_May_2013_KeyResults.txt,Djibouti,May 2013 / May 2013,May 2013,May 2013,Food availability in the Republic of Djibouti ...
495,Madagascar_Sep_2024_-_Aug_2025_KeyResults.txt,Madagascar,Sep 2024 / Aug 2025,Sep 2024,Aug 2025,"Between September and December 2024, around 1...."


In [24]:
import re
import pandas as pd
from gliner import GLiNER
import spacy

# =====================================================================
# 1. INIZIALIZZAZIONE DEI MODELLI IN MEMORIA GLOBALE
# =====================================================================
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")
# Assicurati di aver scaricato il modello: python -m spacy download en_core_web_sm
nlp_model = spacy.load("en_core_web_sm")

# Mappatura dinamica per delegare a GLiNER SOLO le entità spaziali
tag_mapping = {
    "country": "[AFFECTED_AREA]",
    "region": "[AFFECTED_AREA]",
    "province": "[AFFECTED_AREA]",
    "district": "[AFFECTED_AREA]",
    "city": "[AFFECTED_AREA]",
    "village": "[AFFECTED_AREA]",
    "location": "[AFFECTED_AREA]",
    "landmark": "[AFFECTED_AREA]"
}
spatial_labels = list(tag_mapping.keys())


# =====================================================================
# 2. FUNZIONI CORE DI PREPROCESSING E ABLAZIONE
# =====================================================================
def split_into_sentences(text: str, nlp: spacy.Language) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


def apply_regex_abstractions(text: str) -> str:
    """Gestisce le astrazioni temporali e quantitative con priorità corretta."""
    anonymized = text

    # 1. Date composte estese
    anonymized = re.sub(
        r'\b(?:1st|2nd|3rd|\d{1,2}(?:th)?\s+)?(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(?:\d{1,2}(?:st|nd|rd|th)?,?\s+)?\d{4}\b',
        '[DATE]', anonymized, flags=re.IGNORECASE
    )
    # 2. Formati ISO e con separatori
    anonymized = re.sub(r'\b\d{2,4}[-/.]\d{1,2}[-/.]\d{2,4}\b', '[DATE]', anonymized)

    # 3. Mese e Anno
    anonymized = re.sub(
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4}\b',
        '[DATE]', anonymized, flags=re.IGNORECASE
    )
    #mese
    anonymized = re.sub(
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\b',
        '[DATE]', anonymized, flags=re.IGNORECASE
    )
    # 4. Anni isolati
    anonymized = re.sub(r'\b(?:19|20)\d{2}\b', '[DATE]', anonymized)

    # 5. Cifre assolute di popolazione e notazioni testuali estese
    '''
    anonymized = re.sub(
        r'\b(?:\d{1,3}(?:[,\.\s]\d{3})+|\d{4,})(?:\s+(?:million|billion))?\b(?!\s*%)',
        '[POPULATION_FIGURE]', anonymized, flags=re.IGNORECASE
    )
    '''
    return anonymized


def anonymize_sentence(sentence: str, model: GLiNER) -> str:
    # Ablazione Spaziale con GLiNER
    entities = model.predict_entities(sentence, spatial_labels, threshold=0.35, max_len=512)
    entities_sorted = sorted(entities, key=lambda x: x['start'], reverse=True)

    anonymized = sentence
    for ent in entities_sorted:
        start = ent['start']
        end = ent['end']
        placeholder = tag_mapping.get(ent['label'], "[REDACTED]")
        anonymized = anonymized[:start] + placeholder + anonymized[end:]

    # Ablazione Temporale e Quantitativa con RegEx
    return apply_regex_abstractions(anonymized)


# =====================================================================
# 3. ORCHESTRAZIONE E INTEGRAZIONE PANDAS
# =====================================================================
def anonymize_full_report(full_text: str, model: GLiNER, nlp: spacy.Language) -> str:
    if not isinstance(full_text, str) or not full_text.strip():
        return full_text

    # Il modello nlp deve essere iniettato esplicitamente
    sentences = split_into_sentences(full_text, nlp)

    anonymized_sentences = [anonymize_sentence(s, model) for s in sentences]

    return " ".join(anonymized_sentences)


def process_dataframe_safe(df: pd.DataFrame, input_col: str, output_col: str) -> pd.DataFrame:
    print(f"Elaborazione di {len(df)} report con gestione automatica della lunghezza frasi...")

    df[output_col] = df[input_col].apply(
        lambda x: anonymize_full_report(x, gliner_model, nlp_model)
    )

    return df

/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
df_processed = process_dataframe_safe(df, input_col="testo_originale", output_col="gliner_v2")
df_processed

Elaborazione di 497 report con gestione automatica della lunghezza frasi...


KeyboardInterrupt: 

In [30]:
df_processed.to_csv("report_anonimizzati_gliner.csv")

In [27]:
df_processed["gliner_v2"][23]

'During the current period ([DATE] to [DATE]), the [AFFECTED_AREA] of the [AFFECTED_AREA] is experiencing a moderate but sustained [DATE] in acute food security, primarily driven by climatic variability, high food prices, and low food reserves. An estimated 100,000 people (16 percent of the analysed population) are classified in IPC Phase 3 or above (Crisis or worse), requiring urgent action to safeguard livelihoods and address food consumption gaps. This includes approximately 97,000 people in IPC Phase 3 (Crisis) and 3,000 in IPC Phase 4 (Emergency). The [AFFECTED_AREA] is the most severely affected, with 25 percent of its population in Phase 3 or higher, while [AFFECTED_AREA], [AFFECTED_AREA] and [AFFECTED_AREA] remain in Phase 2 (Stressed), despite hosting a significant population in Crisis. Food access and availability are constrained by crop losses caused by prolonged droughts, erratic rainfall, and pest outbreaks, and are further exacerbated by the increasing cost of basic food 

In [28]:
df_processed["gliner"][23]

KeyError: 'gliner'

In [ ]:
df_processed.to_csv("df_duplicati.csv")

In [ ]:
df_processed.paese.unique()

In [ ]:
df_processed["gliner"][1]

In [ ]:
df_processed["testo_originale"][1]